In [1]:
import numpy as np
import pandas as pd

# Data Extracting

In [2]:
df= pd.read_csv("Customer_Dataset.csv")

In [3]:
df

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,No,32,Venmo,Weekly
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,No,41,Bank Transfer,Bi-Weekly
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,No,24,Venmo,Quarterly
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,No,24,Venmo,Weekly


In [4]:
df.shape

(3900, 18)

In [5]:
df.columns

Index(['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category',
       'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season',
       'Review Rating', 'Subscription Status', 'Shipping Type',
       'Discount Applied', 'Promo Code Used', 'Previous Purchases',
       'Payment Method', 'Frequency of Purchases'],
      dtype='str')

# Data Transformation

In [7]:
df.dtypes

Customer ID                       int64
Age                               int64
Gender                              str
Item Purchased                      str
Category                            str
Purchase Amount (USD)             int64
Location                            str
Size                                str
Color                               str
Season                              str
Review Rating                   float64
Subscription Status                 str
Shipping Type                       str
Discount Applied                    str
Promo Code Used                     str
Previous Purchases                int64
Payment Method                      str
Frequency of Purchases              str
purchase_date             datetime64[s]
dtype: object

In [8]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
purchase_date              0
dtype: int64

In [9]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [10]:
df.isnull().sum().sum()

np.int64(0)

In [11]:
df.duplicated().sum()

np.int64(0)

And also there is no any Duplicate Values

In [12]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [13]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases', 'purchase_date'],
      dtype='str')

# Chacking incoorect Values

In [14]:
df["gender"].unique()

<ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str

In [15]:
df["category"].unique()

<ArrowStringArray>
['Clothing', 'Footwear', 'Outerwear', 'Accessories']
Length: 4, dtype: str

In [16]:
df["season"].unique()

<ArrowStringArray>
['Winter', 'Spring', 'Summer', 'Fall']
Length: 4, dtype: str

In [17]:
df["subscription_status"].unique()

<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str

In [18]:
df["promo_code_used"].unique()

<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str

# Feature Engineering

In [19]:
# create a column age group

labels=['Young', 'Adult','Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4,labels=labels)

In [20]:
df[['age', 'age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young
2,50,Middle-aged
3,21,Young
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young
8,26,Young
9,57,Middle-aged


In [21]:
# create new column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [22]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [23]:
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [24]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [25]:
# Dropping promo code used column

df = df.drop('promo_code_used', axis=1)

Monetary

In [26]:
df["monetary"] = (
    df.groupby("customer_id")["purchase_amount"]
      .transform("sum"))

# Loding data to the database

In [33]:
! pip install mysql sqlalchemy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:

from sqlalchemy import create_engine
host="localhost"
user="root"
password="root"
port="3306"
database="customer"

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

table_name = "customer_dataset"
df.to_sql(table_name, engine, if_exists="replace", index=False)

pd.read_sql(f"SELECT * FROM {table_name} LIMIT 10", engine)


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,...,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,purchase_date,age_group,purchase_frequency_days,monetary
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Yes,Express,Yes,14,Venmo,Fortnightly,2024-10-18 21:17:16,Middle-aged,14,53
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Yes,Express,Yes,2,Cash,Fortnightly,2024-07-28 15:21:50,Young,14,64
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Yes,Free Shipping,Yes,23,Credit Card,Weekly,2024-10-20 08:38:02,Middle-aged,7,73
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,Yes,Next Day Air,Yes,49,PayPal,Weekly,2024-11-30 05:23:19,Young,7,90
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,Yes,Free Shipping,Yes,31,PayPal,Annually,2023-09-02 00:03:08,Middle-aged,365,49
5,6,46,Male,Sneakers,Footwear,20,Wyoming,M,White,Summer,...,Yes,Standard,Yes,14,Venmo,Weekly,2023-06-04 02:38:12,Middle-aged,7,20
6,7,63,Male,Shirt,Clothing,85,Montana,M,Gray,Fall,...,Yes,Free Shipping,Yes,49,Cash,Quarterly,2024-02-19 05:22:01,Senior,90,85
7,8,27,Male,Shorts,Clothing,34,Louisiana,L,Charcoal,Winter,...,Yes,Free Shipping,Yes,19,Credit Card,Weekly,2023-11-06 10:37:10,Young,7,34
8,9,26,Male,Coat,Outerwear,97,West Virginia,L,Silver,Summer,...,Yes,Express,Yes,8,Venmo,Annually,2024-07-11 04:23:38,Young,365,97
9,10,57,Male,Handbag,Accessories,31,Missouri,M,Pink,Spring,...,Yes,2-Day Shipping,Yes,4,Cash,Quarterly,2023-11-01 10:04:58,Middle-aged,90,31
